# 01 - baselines

B0, logistic regression and XGBoost on the real feature store, under both splits.
CPU-only really, but it runs here so every number in the paper comes from one environment.

**Read `results/gate1c_b0_tripwire.json` before trusting anything below.** B0 is the
leakage tripwire: comment length plus editor string, nothing about what was edited. It
passed locally at AUC 0.656, which is a weak prior and not a leak. If it ever climbs
above 0.85, stop.

In [ ]:
# Unzip the bundle uploaded through the Altair portal, then work from it.
# Everything real lives in src/mapguard/ - see CONTRIBUTING.md.
import glob, json, os, sys, zipfile, pathlib

BUNDLE = next(iter(sorted(glob.glob("*.zip")) + sorted(glob.glob("/data/**/*.zip", recursive=True))), None)
if BUNDLE and not pathlib.Path("src/mapguard").exists():
    with zipfile.ZipFile(BUNDLE) as z:
        z.extractall(".")
    print("unpacked", BUNDLE)

sys.path.insert(0, "src")
TASK = [p for p in glob.glob("*/BUNDLE_MANIFEST.json")]
if TASK:
    manifest = json.load(open(TASK[0]))
    print(json.dumps({k: manifest[k] for k in ("task","git_sha","git_dirty","generated")}, indent=2))
    if manifest.get("git_dirty"):
        print("WARNING: bundle was built from a dirty tree; git_sha does not fully describe it")

In [ ]:
import json
import numpy as np
import pyarrow.parquet as pq

changesets = pq.read_table("data/interim/changesets.parquet").to_pandas()
print(len(changesets), "changesets x", changesets.shape[1], "columns")
print(changesets["label"].value_counts(dropna=False).to_dict())

## Splits

**R1a** is Ovid's protocol: a random split. It is user-leaky and every table it appears in
must say so. **R1b** is user-disjoint *and class-stratified* - without the stratification
the folds came out 63/23/32 percent positive against 50 overall, because vandals have one
or two changesets each while prolific mappers are overwhelmingly clean.

In [ ]:
from mapguard.evaluation.splits import random_split, user_disjoint_split, assert_user_disjoint

y = changesets["label"].to_numpy(dtype="float32", na_value=0.0)
uids = changesets["uid"].tolist()

r1b = user_disjoint_split(uids, y)
assert_user_disjoint(r1b, uids)
r1a = random_split(len(changesets))

print("R1b user-disjoint:", r1b.summary(y))
print("R1a random (LEAKY):", r1a.summary(y))

## Baselines

Every run records split, seed and git SHA. A number without those cannot go in the paper.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from mapguard.evaluation.metrics import summary as metric_summary, bootstrap_ci
from mapguard.training.dataset import TABULAR_FEATURES

X = np.nan_to_num(changesets[[c for c in TABULAR_FEATURES if c in changesets.columns]]
                  .to_numpy(dtype="float64"), nan=0.0, posinf=0.0, neginf=0.0)

results = {}
for name, split in (("R1b", r1b), ("R1a_leaky", r1a)):
    tr, te = split.train | split.val, split.test
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
    lr.fit(X[tr], y[tr])
    p = lr.predict_proba(X[te])[:, 1]
    res = metric_summary(y[te], p)
    res["pr_auc_ci95"] = bootstrap_ci(y[te], p)
    results[f"logreg_{name}"] = res
    print(name, "PR-AUC", res["by_prevalence"]["0.5"]["pr_auc"], "CI", res["pr_auc_ci95"])

In [ ]:
try:
    import xgboost as xgb
except ImportError:
    !pip install -q xgboost
    import xgboost as xgb

for name, split in (("R1b", r1b), ("R1a_leaky", r1a)):
    tr, te = split.train | split.val, split.test
    clf = xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.08,
                            subsample=0.8, colsample_bytree=0.8, eval_metric="aucpr")
    clf.fit(X[tr], y[tr])
    p = clf.predict_proba(X[te])[:, 1]
    res = metric_summary(y[te], p)
    res["pr_auc_ci95"] = bootstrap_ci(y[te], p)
    results[f"xgboost_{name}"] = res
    print(name, "PR-AUC", res["by_prevalence"]["0.5"]["pr_auc"], "CI", res["pr_auc_ci95"])

## The number that matters

**R1a minus R1b is a finding, not a nuisance.** It measures how much of a published score
is account memorisation rather than edit understanding - a question this literature has
not asked.

In [ ]:
import datetime, os, subprocess
os.makedirs("results", exist_ok=True)
manifest = json.load(open([p for p in __import__("glob").glob("*/BUNDLE_MANIFEST.json")][0]))
out = {
    "generated": datetime.datetime.now().isoformat(timespec="seconds"),
    "git_sha": manifest["git_sha"],
    "feature_schema": manifest.get("feature_schema"),
    "n_changesets": int(len(changesets)),
    "splits": {"R1b": r1b.summary(y), "R1a_leaky": r1a.summary(y)},
    "results": results,
    "leak_note": "R1a is user-leaky and exists only for comparability with Ovid.",
}
json.dump(out, open("results/baselines.json", "w"), indent=2, default=float)
print(json.dumps({k: v for k, v in out.items() if k != "results"}, indent=2))